In [0]:
dbutils.widgets.text("raw_volume_path", "/Volumes/workspace/default/books_raw/", "Raw data volume path")
dbutils.widgets.text("target_schema", "workspace.default", "Target catalog.schema")

raw_volume_path = dbutils.widgets.get("raw_volume_path")
target_schema = dbutils.widgets.get("target_schema")

In [0]:
import logging, sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                     format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("book_etl")

def log_step(step_name, row_count=None, extra=None):
    msg = f"STEP: {step_name}"
    if row_count is not None:
        msg += f" | rows={row_count}"
    if extra:
        msg += f" | {extra}"
    logger.info(msg)

In [0]:
def validate_schema(df, expected_columns, table_name):
    actual = set(df.columns)
    expected = set(expected_columns)
    missing = expected - actual
    if missing:
        raise ValueError(f"[{table_name}] Missing expected columns: {missing}")
    log_step(f"schema_validated_{table_name}", df.count())

In [0]:
validate_schema(df, [
    "book_id", "title", "authors", "average_rating", "ratings_count",
    "genres", "description", "original_publication_year", "pages"
], "books")

2026-07-28 17:08:42,447 | INFO | STEP: schema_validated_books | rows=10000


In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiline", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{raw_volume_path}books_enriched.csv")

In [0]:
display(df.limit(10))
df.count()

_c0,index,authors,average_rating,best_book_id,book_id,books_count,description,genres,goodreads_book_id,image_url,isbn,isbn13,language_code,original_publication_year,original_title,pages,publishDate,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,ratings_count,small_image_url,title,work_id,work_ratings_count,work_text_reviews_count,authors_2
0,0,['Suzanne Collins'],4.34,2767052,1,272,"WINNING MEANS FAME AND FORTUNE.LOSING MEANS CERTAIN DEATH.THE HUNGER GAMES HAVE BEGUN. . . .In the ruins of a place once known as North America lies the nation of Panem, a shining Capitol surrounded by twelve outlying districts. The Capitol is harsh and cruel and keeps the districts in line by forcing them all to send one boy and once girl between the ages of twelve and eighteen to participate in the annual Hunger Games, a fight to the death on live TV.Sixteen-year-old Katniss Everdeen regards it as a death sentence when she steps forward to take her sister's place in the Games. But Katniss has been close to dead before—and survival, for her, is second nature. Without really meaning to, she becomes a contender. But if she is to win, she will have to start making choices that weight survival against humanity and life against love.","['young-adult', 'fiction', 'fantasy', 'science-fiction', 'romance']",2767052,https://images.gr-assets.com/books/1447303603m/2767052.jpg,439023483,9.78043902348E12,eng,2008.0,The Hunger Games,374.0,09/14/08,66715,127936,560092,1481305,2706317,4780653,https://images.gr-assets.com/books/1447303603s/2767052.jpg,"The Hunger Games (The Hunger Games, #1)",2792775,4942365,155254,['Suzanne Collins']
1,1,"['J.K. Rowling', 'Mary GrandPré']",4.44,3,2,491,"Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft and Wizardry.After a lifetime of bottling up his magical powers, Harry finally feels like a normal kid. But even within the Wizarding community, he is special. He is the boy who lived: the only person to have ever survived a killing curse inflicted by the evil Lord Voldemort, who launched a brutal takeover of the Wizarding world, only to vanish after failing to kill Harry.Though Harry's first year at Hogwarts is the best of his life, not everything is perfect. There is a dangerous secret object hidden within the castle walls, and Harry believes it's his responsibility to prevent it from falling into evil hands. But doing so will bring him into contact with forces more terrifying than he ever could have imagined.Full of sympathetic characters, wildly imaginative situations, and countless exciting details, the first installment in the series assembles an unforgettable magical world and sets the stage for many high-stakes adventures to come.","['fantasy', 'fiction', 'young-adult', 'classics']",3,https://images.gr-assets.com/books/1474154022m/3.jpg,439554934,9.78043955493E12,eng,1997.0,Harry Potter and the Philosopher's Stone,309.0,11/01/03,75504,101676,455024,1156318,3011543,4602479,https://images.gr-assets.com/books/1474154022s/3.jpg,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",4640799,4800065,75867,"['J.K. Rowling', 'Mary GrandPré']"
2,2,['Stephenie Meyer'],3.57,41865,3,226,"About three things I was absolutely positive. First, Edward was a vampire. Second, there was a part of him—and I didn't know how dominant that part might be—that thirsted for my blood. And third, I was unconditionally and irrevocably in love with him. Deeply seductive and extraordinarily suspenseful, Twilight is a love story with bite.","['young-adult', 'fantasy', 'romance', 'fiction', 'paranormal']",41865,https://images.gr-assets.com/books/1361039443m/41865.jpg,316015849,9.78031601584E12,eng,2005.0,Twilight,501.0,09/06/06,4561

10000

In [0]:
total = df.count()
unique = df.dropDuplicates().count()
log_step("books_duplicate_check", total, f"unique={unique}, duplicates={total - unique}")

2026-07-28 17:08:45,942 | INFO | STEP: books_duplicate_check | rows=10000 | unique=10000, duplicates=0


In [0]:
missing_desc = df.filter(df.description.isNull()).count()
print(f"Missing descriptions: {missing_desc}")

Missing descriptions: 57


In [0]:
df.describe()

DataFrame[summary: string, _c0: string, index: string, authors: string, average_rating: string, best_book_id: string, book_id: string, books_count: string, description: string, genres: string, goodreads_book_id: string, image_url: string, isbn: string, isbn13: string, language_code: string, original_publication_year: string, original_title: string, pages: string, publishDate: string, ratings_1: string, ratings_2: string, ratings_3: string, ratings_4: string, ratings_5: string, ratings_count: string, small_image_url: string, title: string, work_id: string, work_ratings_count: string, work_text_reviews_count: string, authors_2: string]

In [0]:
display(df.select("_c0", "index", "book_id").limit(10))

_c0,index,book_id
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5
5,5,6
6,6,7
7,7,8
8,8,9
9,9,10


In [0]:
df.filter(df._c0 != df.index).count()

1832

In [0]:
display(df.filter(df._c0 != df.index).select("_c0", "index", "book_id", "title").limit(10))

_c0,index,book_id,title
8167,12,13,1984
8168,13,14,Animal Farm
8169,18,19,"The Fellowship of the Ring (The Lord of the Rings, #1)"
8170,34,35,The Alchemist
8171,43,44,"The Notebook (The Notebook, #1)"
8172,47,48,Fahrenheit 451
8173,70,71,Frankenstein
8174,83,84,"Jurassic Park (Jurassic Park, #1)"
8175,106,107,A Walk to Remember
8176,111,112,"Me Before You (Me Before You, #1)"


In [0]:
df = df.drop("_c0", "index")

In [0]:
display(df.limit(10))

authors,average_rating,best_book_id,book_id,books_count,description,genres,goodreads_book_id,image_url,isbn,isbn13,language_code,original_publication_year,original_title,pages,publishDate,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,ratings_count,small_image_url,title,work_id,work_ratings_count,work_text_reviews_count,authors_2
['Suzanne Collins'],4.34,2767052,1,272,"WINNING MEANS FAME AND FORTUNE.LOSING MEANS CERTAIN DEATH.THE HUNGER GAMES HAVE BEGUN. . . .In the ruins of a place once known as North America lies the nation of Panem, a shining Capitol surrounded by twelve outlying districts. The Capitol is harsh and cruel and keeps the districts in line by forcing them all to send one boy and once girl between the ages of twelve and eighteen to participate in the annual Hunger Games, a fight to the death on live TV.Sixteen-year-old Katniss Everdeen regards it as a death sentence when she steps forward to take her sister's place in the Games. But Katniss has been close to dead before—and survival, for her, is second nature. Without really meaning to, she becomes a contender. But if she is to win, she will have to start making choices that weight survival against humanity and life against love.","['young-adult', 'fiction', 'fantasy', 'science-fiction', 'romance']",2767052,https://images.gr-assets.com/books/1447303603m/2767052.jpg,439023483,9.78043902348E12,eng,2008.0,The Hunger Games,374.0,09/14/08,66715,127936,560092,1481305,2706317,4780653,https://images.gr-assets.com/books/1447303603s/2767052.jpg,"The Hunger Games (The Hunger Games, #1)",2792775,4942365,155254,['Suzanne Collins']
"['J.K. Rowling', 'Mary GrandPré']",4.44,3,2,491,"Harry Potter's life is miserable. His parents are dead and he's stuck with his heartless relatives, who force him to live in a tiny closet under the stairs. But his fortune changes when he receives a letter that tells him the truth about himself: he's a wizard. A mysterious visitor rescues him from his relatives and takes him to his new home, Hogwarts School of Witchcraft and Wizardry.After a lifetime of bottling up his magical powers, Harry finally feels like a normal kid. But even within the Wizarding community, he is special. He is the boy who lived: the only person to have ever survived a killing curse inflicted by the evil Lord Voldemort, who launched a brutal takeover of the Wizarding world, only to vanish after failing to kill Harry.Though Harry's first year at Hogwarts is the best of his life, not everything is perfect. There is a dangerous secret object hidden within the castle walls, and Harry believes it's his responsibility to prevent it from falling into evil hands. But doing so will bring him into contact with forces more terrifying than he ever could have imagined.Full of sympathetic characters, wildly imaginative situations, and countless exciting details, the first installment in the series assembles an unforgettable magical world and sets the stage for many high-stakes adventures to come.","['fantasy', 'fiction', 'young-adult', 'classics']",3,https://images.gr-assets.com/books/1474154022m/3.jpg,439554934,9.78043955493E12,eng,1997.0,Harry Potter and the Philosopher's Stone,309.0,11/01/03,75504,101676,455024,1156318,3011543,4602479,https://images.gr-assets.com/books/1474154022s/3.jpg,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",4640799,4800065,75867,"['J.K. Rowling', 'Mary GrandPré']"
['Stephenie Meyer'],3.57,41865,3,226,"About three things I was absolutely positive. First, Edward was a vampire. Second, there was a part of him—and I didn't know how dominant that part might be—that thirsted for my blood. And third, I was unconditionally and irrevocably in love with him. Deeply seductive and extraordinarily suspenseful, Twilight is a love story with bite.","['young-adult', 'fantasy', 'romance', 'fiction', 'paranormal']",41865,https://images.gr-assets.com/books/1361039443m/41865.jpg,316015849,9.78031601584E12,eng,2005.0,Twilight,501.0,09/06/06,456191,436802,793319,87507

In [0]:
def quality_gate(df, table_name, key_column, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(df[key_column].isNull()).count()
    duplicate_keys = row_count - df.dropDuplicates([key_column]).count()
    log_step(f"quality_check_{table_name}", row_count,
              f"null_keys={null_keys}, duplicate_keys={duplicate_keys}")
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key column '{key_column}' — aborting.")
    if duplicate_keys > 0:
        logger.warning(f"[{table_name}] Found {duplicate_keys} duplicate book_ids — deduplicating.")
        df = df.dropDuplicates([key_column])
    return df

df = quality_gate(df, "books", key_column="book_id", min_expected_rows=9000)

2026-07-28 17:08:51,628 | INFO | STEP: quality_check_books | rows=10000 | null_keys=0, duplicate_keys=0


In [0]:
print(repr(target_schema))

'workspace.default'


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.books")
log_step("write_books_complete", df.count())

2026-07-28 17:08:54,867 | INFO | STEP: write_books_complete | rows=10000


In [0]:
ratings_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_volume_path}ratings.csv")
display(ratings_df.limit(10))
ratings_df.count()

user_id,book_id,rating
1,258,5
2,4081,4
2,260,5
2,9296,5
2,2318,3
2,26,4
2,315,3
2,33,4
2,301,5
2,2686,5


5976479

In [0]:
validate_schema(ratings_df, ["user_id", "book_id", "rating"], "ratings")
log_step("read_ratings", ratings_df.count())

2026-07-28 17:08:59,961 | INFO | STEP: schema_validated_ratings | rows=5976479
2026-07-28 17:09:00,880 | INFO | STEP: read_ratings | rows=5976479


In [0]:
total = ratings_df.count()
unique_pairs = ratings_df.select("user_id", "book_id").distinct().count()
print(f"Total ratings: {total}, unique user/book pairs: {unique_pairs}, duplicates: {total - unique_pairs}")

Total ratings: 5976479, unique user/book pairs: 5976479, duplicates: 0


In [0]:
ratings_df.select("rating").distinct().orderBy("rating").show()

+------+
|rating|
+------+
|     1|
|     2|
|     3|
|     4|
|     5|
+------+



In [0]:
orphans = ratings_df.join(df, "book_id", "left_anti").count()
print(f"Ratings with no matching book: {orphans}")

Ratings with no matching book: 0


In [0]:
def quality_gate_composite(df, table_name, key_columns, min_expected_rows=1):
    row_count = df.count()
    null_keys = df.filter(" OR ".join([f"{c} IS NULL" for c in key_columns])).count()
    duplicate_keys = row_count - df.dropDuplicates(key_columns).count()
    log_step(f"quality_check_{table_name}", row_count,
              f"null_keys={null_keys}, duplicate_keys={duplicate_keys}")
    if row_count < min_expected_rows:
        raise ValueError(f"[{table_name}] Row count {row_count} below minimum {min_expected_rows} — aborting.")
    if null_keys > 0:
        raise ValueError(f"[{table_name}] Found {null_keys} null values in key columns {key_columns} — aborting.")
    if duplicate_keys > 0:
        logger.warning(f"[{table_name}] Found {duplicate_keys} duplicate (user_id, book_id) pairs — deduplicating.")
        df = df.dropDuplicates(key_columns)
    return df

ratings_df = quality_gate_composite(ratings_df, "ratings", key_columns=["user_id", "book_id"], min_expected_rows=900000)

2026-07-28 17:09:09,728 | INFO | STEP: quality_check_ratings | rows=5976479 | null_keys=0, duplicate_keys=0


In [0]:
ratings_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.ratings")
log_step("write_ratings_complete", ratings_df.count())

2026-07-28 17:09:16,987 | INFO | STEP: write_ratings_complete | rows=5976479


In [0]:
book_tags_df = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{raw_volume_path}book_tags.csv")
display(book_tags_df.limit(10))
book_tags_df.count()

goodreads_book_id,tag_id,count
1,30574,167697
1,11305,37174
1,11557,34173
1,8717,12986
1,33114,12716
1,11743,9954
1,14017,7169
1,5207,6221
1,22743,4974
1,32989,4364


999912

In [0]:
validate_schema(book_tags_df, ["goodreads_book_id", "tag_id", "count"], "book_tags")
log_step("read_book_tags", book_tags_df.count())

2026-07-28 17:09:20,630 | INFO | STEP: schema_validated_book_tags | rows=999912
2026-07-28 17:09:21,125 | INFO | STEP: read_book_tags | rows=999912


In [0]:
before = book_tags_df.count()
book_tags_df = book_tags_df.dropDuplicates(["goodreads_book_id", "tag_id"])
after = book_tags_df.count()
log_step("book_tags_deduplicated", after, f"removed={before - after}")

2026-07-28 17:09:22,456 | INFO | STEP: book_tags_deduplicated | rows=999904 | removed=8


In [0]:
orphans = book_tags_df.join(df, "goodreads_book_id", "left_anti").count()
print(f"Tag rows with no matching book: {orphans}")

Tag rows with no matching book: 0


In [0]:
total = book_tags_df.count()
unique_pairs = book_tags_df.select("goodreads_book_id", "tag_id").distinct().count()
print(f"Total: {total}, Unique book/tag pairs: {unique_pairs}, Duplicates: {total - unique_pairs}")

Total: 999904, Unique book/tag pairs: 999904, Duplicates: 0


In [0]:
from pyspark.sql import functions as F
dupe_pairs = book_tags_df.groupBy("goodreads_book_id", "tag_id").count().filter(F.col("count") > 1)
display(dupe_pairs)

goodreads_book_id,tag_id,count


In [0]:
book_tags_df.filter((book_tags_df.goodreads_book_id == 52629) & (book_tags_df.tag_id == 10094)).show()

+-----------------+------+-----+
|goodreads_book_id|tag_id|count|
+-----------------+------+-----+
|            52629| 10094|    1|
+-----------------+------+-----+



In [0]:
book_tags_df = book_tags_df.dropDuplicates(["goodreads_book_id", "tag_id"])
book_tags_df.count()

999904

In [0]:
book_tags_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.book_tags")
log_step("write_book_tags_complete", book_tags_df.count())

2026-07-28 17:09:30,088 | INFO | STEP: write_book_tags_complete | rows=999904
